In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import os
import talib as ta
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

frequency = "1d"
window_pred = 7
# Cargar los datos para esta frecuencia
BTCUSDT = pd.read_csv('Datos/BTCUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
AVAXUSDT = pd.read_csv('Datos/AVAXUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
XRPUSDT = pd.read_csv('Datos/XRPUSDT_1d_01-01-2016_01-01-2025.csv', index_col='timestamp')
data = pd.DataFrame() 

data['BTCUSDT'] = BTCUSDT[['close']]
data['AVAXUSDT'] = AVAXUSDT[['close']]
data['XRPUSDT'] = XRPUSDT[['close']]
data.head()

,BTCUSDT,AVAXUSDT,XRPUSDT
timestamp,,,
2017-08-17,4285.08,NaN,NaN
2017-08-18,4108.37,NaN,NaN
2017-08-19,4139.98,NaN,NaN
2017-08-20,4086.29,NaN,NaN
2017-08-21,4016.00,NaN,NaN


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [2]:
def save_results_global(model, crypto, acc, sample, frequency='1d'):
    file_name = f'accuracy_results_{frequency}_glob.csv'

    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'What'])

    new_row = pd.DataFrame([[model, crypto, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'What'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)
    df_results.to_csv(file_name, index=False)

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [3]:
def charact_lags(data, ric, lags, window_pred, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df[ric].pct_change(window) #momentum de la ventana pct_change(12)
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df = df.iloc[:-window_pred]
    df['d'] = np.where(df[ric].shift(-window_pred) > df[ric], 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán 
    features = [ric, 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    return df, cols

lags = 5

dfs = {}
for ric in data:
    df, cols = charact_lags(data, ric, lags, window_pred)
    dfs[ric] = df, cols


In [4]:
#comprueba si los datos están desbalanceados 
df_d = pd.DataFrame()
for ric in data:
    df, cols = dfs[ric]
    df_d[ric] = df['d'].value_counts(normalize=True)
df_d

,BTCUSDT,AVAXUSDT,XRPUSDT
d,,,
1,0.536458,0.28125,0.425595
0,0.463542,0.71875,0.574405


Comentar que he mirado si los datos están desbalanceados 

In [5]:
dfs[ric][0]

,XRPUSDT,r,sma,min,max,mom,vol,rsi,atr,d,...,rsi_lag_1,rsi_lag_2,rsi_lag_3,rsi_lag_4,rsi_lag_5,atr_lag_1,atr_lag_2,atr_lag_3,atr_lag_4,atr_lag_5
timestamp,,,,,,,,,,,,,,,,,,,,,
2017-08-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-08-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-21,2.2380,-0.018286,2.147353,1.3984,2.7245,0.794707,0.078791,61.230947,0.087947,0,...,62.204662,61.629097,63.277124,69.778755,68.774720,0.089555,0.091254,0.091942,0.086250,0.086355
2024-12-22,2.2031,-0.015717,2.171703,1.3984,2.7245,0.496061,0.073957,60.404383,0.086179,0,...,61.230947,62.204662,61.629097,63.277124,69.778755,0.087947,0.089555,0.091254,0.091942,0.086250
2024-12-23,2.2617,0.026251,2.198143,1.3984,2.7245,0.540143,0.073927,61.311540,0.085259,0,...,60.404383,61.230947,62.204662,61.629097,63.277124,0.086179,0.087947,0.089555,0.091254,0.091942


Cambio el formato de los datos para que funcione con el modelo global

In [6]:

cryptos = list(dfs.keys())

df_global = []

for ric, (df, cols) in dfs.items():
    df = df.copy().reset_index()
    df['crypto'] = ric
    df.rename(columns={ric: 'close'}, inplace=True)

    # Renombrar columnas tipo 'BTCUSDT_lag_1' -> 'close_lag_1'
    lag_cols = {f'{ric}_lag_{i}': f'close_lag_{i}' for i in range(1, lags + 1)}
    df.rename(columns=lag_cols, inplace=True)

    df_global.append(df)

# Concatenar todo en un solo DataFrame
df_global = pd.concat(df_global, ignore_index=True)

# Ordenar por fecha
df_global = df_global.sort_values(by='timestamp').reset_index(drop=True)

df_global.dropna(inplace=True)
df_global

,timestamp,close,r,sma,min,max,mom,vol,rsi,atr,...,rsi_lag_2,rsi_lag_3,rsi_lag_4,rsi_lag_5,atr_lag_1,atr_lag_2,atr_lag_3,atr_lag_4,atr_lag_5,crypto
105,2017-09-21,3609.9900,-0.077272,4198.476667,3189.0200,4834.9100,-0.106438,0.065332,43.781986,163.012627,...,46.640426,47.857836,44.074957,44.213417,158.633407,163.757318,165.094811,159.235322,164.210333,BTCUSDT
108,2017-09-22,3595.8700,-0.003919,4181.205333,3189.0200,4834.9100,-0.125945,0.065201,43.651605,158.049539,...,46.542029,46.640426,47.857836,44.074957,163.012627,158.633407,163.757318,165.094811,159.235322,BTCUSDT
113,2017-09-23,3780.0000,0.049938,4163.338333,3189.0200,4834.9100,-0.124191,0.065257,45.827858,158.918888,...,43.781986,46.542029,46.640426,47.857836,158.049539,163.012627,158.633407,163.757318,165.094811,BTCUSDT
115,2017-09-24,3660.0200,-0.032255,4142.649667,3189.0200,4834.9100,-0.144991,0.065453,44.665062,157.620925,...,43.651605,43.781986,46.542029,46.640426,158.918888,158.049539,163.012627,158.633407,163.757318,BTCUSDT
118,2017-09-25,3920.7500,0.068814,4128.760000,3189.0200,4834.9100,-0.096068,0.066767,47.651044,161.057894,...,45.827858,43.651605,43.781986,46.542029,157.620925,158.918888,158.049539,163.012627,158.633407,BTCUSDT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8059,2024-12-24,98663.5800,0.039087,98381.171000,91965.1600,106133.7400,0.007799,0.025230,57.773555,1730.867671,...,54.791636,57.141430,57.724947,57.443840,1660.135177,1706.870873,1693.117454,1733.795642,1781.734113,BTCUSDT
8060,2024-12-24,41.2500,0.053273,46.232000,36.5700,53.9800,-0.019258,0.065316,51.054260,1.953744,...,46.890348,47.590124,49.646051,48.742458,1.947322,1.926885,1.964018,1.947605,1.979247,AVAXUSDT
8061,2024-12-25,40.2500,-0.024541,46.196667,36.5700,53.9800,-0.025660,0.065387,50.168802,1.921953,...,49.199477,46.890348,47.590124,49.646051,1.953744,1.947322,1.926885,1.964018,1.947605,AVAXUSDT
8062,2024-12-25,99429.6000,0.007734,98595.157333,91965.1600,106133.7400,0.069020,0.023303,58.408279,1698.706082,...,54.456312,54.791636,57.141430,57.724947,1730.867671,1660.135177,1706.870873,1693.117454,1733.795642,BTCUSDT


In [7]:
# Prueba a entrenar sin la columna d_lag_n para ver la importancia que tiene
'''
X_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])
y_train = train['d']
X_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])
y_test = test['d']
'''

"\nX_train = train.drop(columns=['d', 'timestamp'] + [col for col in train.columns if col.startswith('d_lag_')])\ny_train = train['d']\nX_test = test.drop(columns=['d', 'timestamp'] + [col for col in test.columns if col.startswith('d_lag_')])\ny_test = test['d']\n"

Modelo MLP Classifier GLOBAL

In [ ]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

            if len(test) == 0:
                continue

            drop_cols = ['d', 'timestamp', 'crypto', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 
                 'rsi', 'atr'] + [col for col in train.columns if 'close' in col]
            X_train, y_train = train.drop(columns=drop_cols), train['d']
            X_test, y_test = test.drop(columns=drop_cols), test['d']

            # Normalización 
            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)

            # Obtención de la importancia de las características
            result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")
            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')
        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d', 'timestamp', 'crypto', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 
                 'rsi', 'atr'] + [col for col in train.columns if 'close' in col]
    X_train, y_train = train.drop(columns=drop_cols), train['d']
    X_test, y_test = test.drop(columns=drop_cols), test['d']

    # Normalización
    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    # Obtención de la importancia de las características
    result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
    sorted_idx = result.importances_mean.argsort()[::-1]
    print("Feature importances (top 10):")
    for i in sorted_idx[:10]:
        print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")
        
    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f} | f1={f1:.4f}')
    save_results_global(model_class.__name__, "global", acc, "FINAL TEST", frequency=freq)
    save_results_global(model_class.__name__, "global", f1, "F1 FINAL TEST", frequency=freq)
    
    return best_params

Cambios hablados 6/5: normalización y eliminación de close_lags_x
reservo crypto para imprimir los resultados pero no la uso para entrenar

In [ ]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def normalize_with_close(X, close_col):
                ratio_cols = [col for col in X.columns if any(x in col for x in ['sma', 'atr', 'min', 'max'])]
                for col in ratio_cols:
                    X[col] = X[col] / close_col
                X.drop(columns=[col for col in X.columns if 'close' in col], inplace=True) # eliminar close para que no esté en los datos de entrenamiento
                return X
    
    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []
        crypto_accs = {}

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

            if len(test) == 0:
                continue

            drop_cols = ['d', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 
                         'rsi', 'atr'] # close... se eliminan después de normalizar los datos con close_lag_1
            X_train, y_train = train.drop(columns=drop_cols), train['d']
            X_test, y_test = test.drop(columns=drop_cols), test['d']
            
            # Normalizar con close
            close_train, close_test = train['close_lag_1'], test['close_lag_1']
            X_train = normalize_with_close(X_train.copy(), close_train)
            X_test = normalize_with_close(X_test.copy(), close_test)

            # timestamp y crypto como índice
            X_train = X_train.set_index(['crypto', 'timestamp'])
            X_test = X_test.set_index(['crypto', 'timestamp'])

            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)

            # Importancia de cada característica
            result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")

            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

        df_results_test = X_test.copy()
        df_results_test['true'] = y_test.values
        df_results_test['pred'] = pred
        accuracy_per_crypto = df_results_test.groupby(level='crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))

        for crypto, acc_c in accuracy_per_crypto.items():
            crypto_accs.setdefault(crypto, []).append(acc_c)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')

        print("\nAccuracy promedio por criptomoneda (VAL):")
        for crypto, acc_list in crypto_accs.items():
            avg_crypto_acc = np.mean(acc_list)
            print(f"{crypto:<15} | acc = {avg_crypto_acc:.4f}")
            save_results_global(model_class.__name__, crypto, avg_crypto_acc, "Val", frequency=freq)

        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 
                 'rsi', 'atr'] # close... se eliminan después de normalizar los datos con close_lag_1
    X_train, y_train = train.drop(columns=drop_cols), train['d']
    X_test, y_test = test.drop(columns=drop_cols), test['d']

    # NORMALIZAR CON CLOSE
    close_train, close_test = train['close_lag_1'], test['close_lag_1']
    X_train = normalize_with_close(X_train.copy(), close_train)
    X_test = normalize_with_close(X_test.copy(), close_test)

    # timestamp y crypto como índice
    X_train = X_train.set_index(['crypto', 'timestamp'])
    X_test = X_test.set_index(['crypto', 'timestamp'])


    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    # Importancia de cada característica
    result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
    sorted_idx = result.importances_mean.argsort()[::-1]
    print("Feature importances (top 10):")
    for i in sorted_idx[:10]:
        print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f}')
    save_results_global(model_class.__name__, "global", acc, "FINAL TEST", frequency=freq)

    # Accuracy por criptomoneda final
    df_results_test = X_test.copy()
    df_results_test['true'] = y_test.values
    df_results_test['pred'] = pred
    
    print("\nFINAL TEST - Accuracy y F1 por criptomoneda:")
    grouped = df_results_test.groupby('crypto')
    for crypto, group in grouped:
        acc_c = accuracy_score(group['true'], group['pred'])
        f1_c = f1_score(group['true'], group['pred'], zero_division=0)  # evita error si solo hay una clase
        print(f"{crypto:<15} | acc = {acc_c:.4f} | f1 = {f1_c:.4f}")
        save_results_global(model_class.__name__, crypto, acc_c, "CRYPTO TEST", frequency=freq)
        save_results_global(model_class.__name__, crypto, f1_c, "F1 CRYPTO TEST", frequency=freq)


Incluimos la columna crypto haciendo un Dummies

In [ ]:
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def normalize_with_close(X, close_col):
        ratio_cols = [col for col in X.columns if any(x in col for x in ['sma', 'atr', 'min', 'max'])]
        for col in ratio_cols:
            X[col] = X[col] / close_col
        X.drop(columns=[col for col in X.columns if 'close' in col], inplace=True) # eliminar close para que no esté en los datos de entrenamiento
        return X

    def prepare_features(df):
        df = df.copy()
        crypto_dummies = pd.get_dummies(df['crypto'], prefix='crypto')
        df = pd.concat([df.drop(columns=['crypto']), crypto_dummies], axis=1)
        return df, crypto_dummies.columns

    def objective(trial):
        trial_params = {
            "hidden_layer_sizes": (trial.suggest_int("hidden_units", 32, 1024, step=64),),
            "alpha": trial.suggest_loguniform("alpha", 1e-5, 1e-1),
            "learning_rate_init": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
            "max_iter": model_params.get("max_iter", 1000),
            "early_stopping": model_params.get("early_stopping", True),
            "validation_fraction": model_params.get("validation_fraction", 0.15),
            "shuffle": model_params.get("shuffle", False),
            "random_state": model_params.get("random_state", 100),
        }

        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current_time = min_time + period
        while current_time < cutoff:
            split_dates.append(current_time)
            current_time += period
        split_dates = split_dates[-5:]

        results = []
        crypto_accs = {}

        for split_date in split_dates:
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test = df_trainval[(df_trainval['timestamp'] >= split_date) & (df_trainval['timestamp'] < split_date + period)]

            if len(test) == 0:
                continue

            drop_cols = ['d', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 
                         'rsi', 'atr'] # close... se eliminan después de normalizar los datos con close_lag_1
            X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
            X_test_raw, y_test = test.drop(columns=drop_cols), test['d']
            
            # Guardar criptos antes de codificar
            cryptos_test = test['crypto'].values

            # Normalizar con close
            close_train, close_test = train['close_lag_1'], test['close_lag_1']
            X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
            X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

            # One-hot encoding
            X_train, _ = prepare_features(X_train_raw)
            X_test, _ = prepare_features(X_test_raw)

            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std

            model = model_class(**trial_params)
            model.fit(X_train, y_train)

            # Importancia de cada característica
            result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=0)
            sorted_idx = result.importances_mean.argsort()[::-1]
            print("Feature importances (top 10):")
            for i in sorted_idx[:10]:
                print(f"{X_train.columns[i]:<30} - Importance: {result.importances_mean[i]:.4f}")

            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)
            #print(f'desbalanceo en train {pd.Series(pred).value_counts(normalize=True)}')
            
            df_results_test = X_test.copy()
            df_results_test['true'] = y_test.values
            df_results_test['pred'] = pred
            df_results_test['crypto'] = cryptos_test

            accuracy_per_crypto = df_results_test.groupby('crypto').apply(lambda g: accuracy_score(g['true'], g['pred']))
            for crypto, acc_c in accuracy_per_crypto.items():
                crypto_accs.setdefault(crypto, []).append(acc_c)

        avg_acc = np.mean(results)
        print(f'VALIDATION | acc={avg_acc:.4f}')

        print("\nAccuracy promedio por criptomoneda (VAL):")
        for crypto, acc_list in crypto_accs.items():
            avg_crypto_acc = np.mean(acc_list)
            print(f"{crypto:<15} | acc = {avg_crypto_acc:.4f}")
            save_results_global(model_class.__name__, crypto, avg_crypto_acc, "Val", frequency=freq)

        save_results_global(model_class.__name__, "global", avg_acc, "Val", frequency=freq)
        return avg_acc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 
                  'rsi', 'atr'] # close... se eliminan después de normalizar los datos con close_lag_1
    X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
    X_test_raw, y_test = test.drop(columns=drop_cols), test['d']
    cryptos_test = test['crypto'].values

    # Normalizar
    close_train, close_test = train['close_lag_1'], test['close_lag_1']
    X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
    X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

    # One-hot encoding
    X_train, _ = prepare_features(X_train_raw)
    X_test, _ = prepare_features(X_test_raw)

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    model = model_class(
        hidden_layer_sizes=(best_params["hidden_units"],),
        alpha=best_params["alpha"],
        learning_rate_init=best_params["learning_rate"],
        max_iter=model_params.get("max_iter", 1000),
        early_stopping=model_params.get("early_stopping", True),
        validation_fraction=model_params.get("validation_fraction", 0.15),
        shuffle=model_params.get("shuffle", False),
        random_state=model_params.get("random_state", 100),
    )
    model.fit(X_train, y_train)

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f}')
    print(f'desbalanceo de los datos reales {y_test.value_counts(normalize=True)}')
    print(f'desbalanceo de las predicciones {pd.Series(pred).value_counts(normalize=True)}')
    save_results_global(model_class.__name__, "global", acc, "FINAL TEST", frequency=freq)

    # Accuracy por criptomoneda final
    df_results_test = X_test.copy()
    df_results_test['true'] = y_test.values
    df_results_test['pred'] = pred
    df_results_test['crypto'] = cryptos_test
    
    print("\nFINAL TEST - Accuracy y F1 por criptomoneda:")
    grouped = df_results_test.groupby('crypto')
    for crypto, group in grouped:
        acc_c = accuracy_score(group['true'], group['pred'])
        f1_c = f1_score(group['true'], group['pred']) 
        print(f"{crypto:<15} | acc = {acc_c:.4f} | f1 = {f1_c:.4f}")
        save_results_global(model_class.__name__, crypto, acc_c, "CRYPTO TEST", frequency=freq)
        save_results_global(model_class.__name__, crypto, f1_c, "F1 CRYPTO TEST", frequency=freq)
    
    return best_params

Todos los Modelos

In [ ]:
def normalize_with_close(X, close_col):
    """
    Normaliza columnas ratio en función del precio de cierre.
    """
    ratio_cols = [col for col in X.columns if any(x in col for x in ['sma','atr','min','max'])]
    for col in ratio_cols:
        X[col] = X[col] / close_col
    return X

# ---------------------------------------------------
def prepare_features(df):
    """
    One-hot encoding de la columna 'crypto'.
    """
    crypto_dummies = pd.get_dummies(df['crypto'], prefix='crypto')
    X = pd.concat([df.drop(columns=['crypto']), crypto_dummies], axis=1)
    return X, crypto_dummies.columns

# ---------------------------------------------------
def walk_forward_fit_test(model_class, data, freq, model_params={}, n_trials=5):
    """
    Validación walk-forward y test final para distintos modelos.
    Devuelve: best_params

    Para acelerar, preprocesamos y guardamos los splits una sola vez fuera de la función objetivo.
    """
    # Definir periodos según frecuencia
    if freq == '1h': period = pd.Timedelta(days=7)
    elif freq == '4h': period = pd.Timedelta(days=15)
    else: period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    # Preprocesar todo el dataset una vez
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    # Separar validación y test final
    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period
    df_trainval = df[df['timestamp'] < cutoff]

    # Generar últimos 5 splits walk-forward
    min_time = df_trainval['timestamp'].min()
    split_dates = []
    cur = min_time + period
    while cur < cutoff:
        split_dates.append(cur)
        cur += period
    split_dates = split_dates[-5:]

    # Preprocesar y almacenar splits para acelerar trials
    pre_splits = []
    for sd in split_dates:
        train = df_trainval[df_trainval['timestamp'] < (sd - pd.Timedelta(days=window_pred))]
        test  = df_trainval[(df_trainval['timestamp'] >= sd) & (df_trainval['timestamp'] < sd + period)]
        if test.empty: continue

        drop_cols = ['d','timestamp','close','r','sma','min','max','mom','vol','rsi','atr'] + [c for c in train.columns if 'close' in c]
        X_tr = train.drop(columns=drop_cols)
        X_te = test.drop(columns=drop_cols)
        y_tr = train['d'].values
        y_te = test['d'].values

        # Normalizar
        X_tr = normalize_with_close(X_tr.copy(), train['close'])
        X_te = normalize_with_close(X_te.copy(), test['close'])
        # One-hot + Z-score
        X_tr, _ = prepare_features(X_tr)
        X_te, _ = prepare_features(X_te)
        m, s = X_tr.mean(), X_tr.std().replace(0,1)
        X_tr = (X_tr - m) / s
        X_te = (X_te - m) / s

        pre_splits.append((X_tr.values, X_te.values, y_tr, y_te))

    # Definir sugerencias de hiperparámetros
    def suggest_params(trial):
        if model_class is RandomForestClassifier:
            return {
                'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
                'max_depth': trial.suggest_int('max_depth', 3, 30),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt','log2',None]),
                'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
                'random_state': 42,
                'n_jobs': -1
            }
        elif model_class is GradientBoostingClassifier:
            return {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'random_state': 42
            }
        elif model_class is MLPClassifier:
            return {
                'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(50,),(100,),(100,50)]),
                'activation': trial.suggest_categorical('activation', ['relu','tanh']),
                'solver': trial.suggest_categorical('solver', ['adam','sgd']),
                'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
                'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-3, 1e-1),
                'max_iter': 1000,
                'random_state': 42
            }
        elif model_class is SVC:
            return {
                'C': trial.suggest_float('C', 0.1, 100.0, log=True),
                'gamma': trial.suggest_float('gamma', 1e-4, 1e-1, log=True),
                'kernel': trial.suggest_categorical('kernel', ['rbf','poly','sigmoid'])
            }
        else:
            raise ValueError('Modelo no soportado')

    # Función objetivo simplificada que reutiliza pre_splits
    def objective(trial):
        params = suggest_params(trial)
        f1_list = []
        for X_tr, X_te, y_tr, y_te in pre_splits:
            model = model_class(**params)
            if model_class is GradientBoostingClassifier:
                w = compute_sample_weight(class_weight='balanced', y=y_tr)
                model.fit(X_tr, y_tr, sample_weight=w)
            else:
                model.fit(X_tr, y_tr)
            preds = (model.predict(X_te) > 0.5).astype(int)
            f1_list.append(f1_score(y_te, preds, zero_division=0))
        return np.mean(f1_list)

    # Optimización de hiperparámetros
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)
    best_params = study.best_params

    # Entrenamiento final y test completo
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period
    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test  = df[df['timestamp'] >= cutoff]
    if test.empty: return best_params

    drop_cols = ['d','timestamp','close','r','sma','min','max','mom','vol','rsi','atr'] + \
                [c for c in train.columns if 'close' in c]
    X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
    X_test_raw,  y_test  = test .drop(columns=drop_cols), test ['d']
    crypto_labels = test['crypto'].values

    X_train_raw = normalize_with_close(X_train_raw.copy(), train['close'])
    X_test_raw  = normalize_with_close(X_test_raw.copy(),  test ['close'])
    X_train, _  = prepare_features(X_train_raw)
    X_test, _   = prepare_features(X_test_raw)
    m, s = X_train.mean(), X_train.std(); s.replace(0,1,inplace=True)
    X_train, X_test = (X_train-m)/s, (X_test-m)/s

    final_model = model_class(**best_params)
    if model_class is GradientBoostingClassifier:
        sw = compute_sample_weight(class_weight='balanced', y=y_train)
        final_model.fit(X_train, y_train, sample_weight=sw)
    else:
        final_model.fit(X_train, y_train)

    preds = (final_model.predict(X_test) > 0.5).astype(int)
    acc, f1 = accuracy_score(y_test, preds), f1_score(y_test, preds, zero_division=0)
    print(f'FINAL TEST | acc={acc:.4f} | f1={f1:.4f}')
    print('Desbalanceo reales:', y_test.value_counts(normalize=True).to_dict())
    print('Desbalanceo predicciones:', pd.Series(preds).value_counts(normalize=True).to_dict())

    save_results_global(model_class.__name__, 'global', acc, 'FINAL TEST', frequency=freq)
    save_results_global(model_class.__name__, 'global', f1,  'F1 FINAL TEST', frequency=freq)

    # Métricas por criptomoneda
    df_res = pd.DataFrame({'true': y_test, 'pred': preds, 'crypto': crypto_labels})
    print('\nFINAL TEST - por criptomoneda:')
    for cr, grp in df_res.groupby('crypto'):
        acc_c = accuracy_score(grp['true'], grp['pred'])
        f1_c  = f1_score(grp['true'], grp['pred'], zero_division=0)
        print(f"{cr:<10} | acc={acc_c:.4f} | f1={f1_c:.4f}")

        # Desbalanceo por criptomoneda
        real_dist = grp['true'].value_counts(normalize=True).to_dict()
        pred_dist = grp['pred'].value_counts(normalize=True).to_dict()
        print(f"    Desbalanceo reales      : {real_dist}")
        print(f"    Desbalanceo predicciones: {pred_dist}")

        save_results_global(model_class.__name__, cr, acc_c, 'CRYPTO TEST', frequency=freq)
        save_results_global(model_class.__name__, cr, f1_c,  'F1 CRYPTO TEST', frequency=freq)

    return best_params


In [ ]:
# Lista de modelos a evaluar
modelos = [
    RandomForestClassifier,
    GradientBoostingClassifier,
    MLPClassifier,
    SVC
]

# Parámetros por defecto para algunos modelos
default_params = {
    MLPClassifier: {
        "max_iter": 1000,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "shuffle": False,
        "random_state": 100
    },
    SVC: {
        "max_iter": -1
    }
}

# Ejecutar walk_forward_fit_test para cada modelo
for modelo in modelos:
    print(f"\n=== Ejecutando modelo: {modelo.__name__} ===")
    params = default_params.get(modelo, {})
    best = walk_forward_fit_test(modelo, df_global, frequency, model_params=params, n_trials=5)
    print(f"Mejores parámetros para {modelo.__name__}: {best}")



=== Ejecutando modelo: RandomForestClassifier ===


[I 2025-06-01 11:24:22,231] A new study created in memory with name: no-name-9f57e64d-281e-4d16-b629-7c7a3cea78b7
[I 2025-06-01 11:24:56,397] Trial 4 finished with value: 0.6992571160977206 and parameters: {'n_estimators': 100, 'max_depth': 11, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 4 with value: 0.6992571160977206.
[I 2025-06-01 11:25:16,481] Trial 1 finished with value: 0.7139256216839464 and parameters: {'n_estimators': 600, 'max_depth': 3, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': False}. Best is trial 1 with value: 0.7139256216839464.
[I 2025-06-01 11:25:31,291] Trial 2 finished with value: 0.6422334049312807 and parameters: {'n_estimators': 100, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': None, 'bootstrap': False}. Best is trial 1 with value: 0.7139256216839464.
[I 2025-06-01 11:26:01,030] Trial 0 finished with value: 0.6948088942521301 an

FINAL TEST | acc=0.6885 | f1=0.7206
Desbalanceo reales: {0: 0.5081967213114754, 1: 0.4918032786885246}
Desbalanceo predicciones: {1: 0.6229508196721312, 0: 0.3770491803278688}

FINAL TEST - por criptomoneda:
AVAXUSDT   | acc=0.4590 | f1=0.5769
    Desbalanceo reales      : {0: 0.5901639344262295, 1: 0.4098360655737705}
    Desbalanceo predicciones: {1: 0.8688524590163934, 0: 0.13114754098360656}
BTCUSDT    | acc=0.7705 | f1=0.0000
    Desbalanceo reales      : {0: 0.7704918032786885, 1: 0.22950819672131148}
    Desbalanceo predicciones: {0: 1.0}
XRPUSDT    | acc=0.8361 | f1=0.9107
    Desbalanceo reales      : {1: 0.8360655737704918, 0: 0.16393442622950818}
    Desbalanceo predicciones: {1: 1.0}
Mejores parámetros para RandomForestClassifier: {'n_estimators': 600, 'max_depth': 3, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'log2', 'bootstrap': False}

=== Ejecutando modelo: GradientBoostingClassifier ===


[I 2025-06-01 11:26:18,586] A new study created in memory with name: no-name-39973169-ec77-4324-8782-da2925c6b3c4
[I 2025-06-01 11:27:27,697] Trial 0 finished with value: 0.6903400710107843 and parameters: {'n_estimators': 60, 'max_depth': 4, 'learning_rate': 0.055988067921330216, 'min_samples_split': 4, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.6903400710107843.
[I 2025-06-01 11:29:32,301] Trial 4 finished with value: 0.659179624987712 and parameters: {'n_estimators': 185, 'max_depth': 4, 'learning_rate': 0.1720553958145048, 'min_samples_split': 10, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.6903400710107843.
[I 2025-06-01 11:29:40,621] Trial 3 finished with value: 0.6689514265956433 and parameters: {'n_estimators': 191, 'max_depth': 4, 'learning_rate': 0.16535383163704123, 'min_samples_split': 7, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.6903400710107843.
[I 2025-06-01 11:32:49,448] Trial 2 finished with value: 0.6540852244783433 and parameters: {'

FINAL TEST | acc=0.7077 | f1=0.6905
Desbalanceo reales: {0: 0.5081967213114754, 1: 0.4918032786885246}
Desbalanceo predicciones: {0: 0.5473588342440802, 1: 0.45264116575591984}

FINAL TEST - por criptomoneda:
AVAXUSDT   | acc=0.5191 | f1=0.3759
    Desbalanceo reales      : {0: 0.5901639344262295, 1: 0.4098360655737705}
    Desbalanceo predicciones: {0: 0.639344262295082, 1: 0.36065573770491804}
BTCUSDT    | acc=0.7705 | f1=0.0000
    Desbalanceo reales      : {0: 0.7704918032786885, 1: 0.22950819672131148}
    Desbalanceo predicciones: {0: 1.0}
XRPUSDT    | acc=0.8333 | f1=0.9091
    Desbalanceo reales      : {1: 0.8360655737704918, 0: 0.16393442622950818}
    Desbalanceo predicciones: {1: 0.9972677595628415, 0: 0.00273224043715847}
Mejores parámetros para GradientBoostingClassifier: {'n_estimators': 60, 'max_depth': 4, 'learning_rate': 0.055988067921330216, 'min_samples_split': 4, 'min_samples_leaf': 8}

=== Ejecutando modelo: MLPClassifier ===


[I 2025-06-01 11:35:46,820] A new study created in memory with name: no-name-e03ced43-65cd-43a7-a8dd-cdfd06bbabd9
C:\Users\raque\AppData\Roaming\Python\Python311\site-packages\optuna\distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (50,) which is of type tuple.
  warnings.warn(message)
C:\Users\raque\AppData\Roaming\Python\Python311\site-packages\optuna\distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (100,) which is of type tuple.
  warnings.warn(message)
C:\Users\raque\AppData\Roaming\Python\Python311\site-packages\optuna\distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (100, 50) which is of type tuple.
  warnings.warn(message)
C:\Users\raque\AppData\

FINAL TEST | acc=0.7140 | f1=0.6866
Desbalanceo reales: {0: 0.5081967213114754, 1: 0.4918032786885246}
Desbalanceo predicciones: {0: 0.5792349726775956, 1: 0.4207650273224044}

FINAL TEST - por criptomoneda:
AVAXUSDT   | acc=0.5464 | f1=0.3360
    Desbalanceo reales      : {0: 0.5901639344262295, 1: 0.4098360655737705}
    Desbalanceo predicciones: {0: 0.726775956284153, 1: 0.273224043715847}
BTCUSDT    | acc=0.7705 | f1=0.0000
    Desbalanceo reales      : {0: 0.7704918032786885, 1: 0.22950819672131148}
    Desbalanceo predicciones: {0: 1.0}
XRPUSDT    | acc=0.8251 | f1=0.9042
    Desbalanceo reales      : {1: 0.8360655737704918, 0: 0.16393442622950818}
    Desbalanceo predicciones: {1: 0.9890710382513661, 0: 0.01092896174863388}
Mejores parámetros para MLPClassifier: {'hidden_layer_sizes': (50,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0059471985704823436, 'learning_rate_init': 0.02594415720883459}

=== Ejecutando modelo: SVC ===


[I 2025-06-01 11:38:50,121] A new study created in memory with name: no-name-8e59c8f4-3f45-4a91-8fcb-3e24ddb64060
[I 2025-06-01 11:39:15,584] Trial 2 finished with value: 0.6830080369010035 and parameters: {'C': 1.1637659309286268, 'gamma': 0.004906522596189672, 'kernel': 'rbf'}. Best is trial 2 with value: 0.6830080369010035.
[I 2025-06-01 11:39:16,392] Trial 4 finished with value: 0.6739042654027914 and parameters: {'C': 4.819220222427212, 'gamma': 0.0001418860023691283, 'kernel': 'poly'}. Best is trial 2 with value: 0.6830080369010035.
[I 2025-06-01 11:39:19,001] Trial 3 finished with value: 0.6875589891051741 and parameters: {'C': 12.316339982367632, 'gamma': 0.001401858802201365, 'kernel': 'rbf'}. Best is trial 3 with value: 0.6875589891051741.
[I 2025-06-01 11:39:19,180] Trial 1 finished with value: 0.7555531193680401 and parameters: {'C': 12.957411014425523, 'gamma': 0.0001494632984743774, 'kernel': 'sigmoid'}. Best is trial 1 with value: 0.7555531193680401.
[I 2025-06-01 11:39:

FINAL TEST | acc=0.6721 | f1=0.7170
Desbalanceo reales: {0: 0.5081967213114754, 1: 0.4918032786885246}
Desbalanceo predicciones: {1: 0.6666666666666666, 0: 0.3333333333333333}

FINAL TEST - por criptomoneda:
AVAXUSDT   | acc=0.4098 | f1=0.5814
    Desbalanceo reales      : {0: 0.5901639344262295, 1: 0.4098360655737705}
    Desbalanceo predicciones: {1: 1.0}
BTCUSDT    | acc=0.7705 | f1=0.0000
    Desbalanceo reales      : {0: 0.7704918032786885, 1: 0.22950819672131148}
    Desbalanceo predicciones: {0: 1.0}
XRPUSDT    | acc=0.8361 | f1=0.9107
    Desbalanceo reales      : {1: 0.8360655737704918, 0: 0.16393442622950818}
    Desbalanceo predicciones: {1: 1.0}
Mejores parámetros para SVC: {'C': 12.957411014425523, 'gamma': 0.0001494632984743774, 'kernel': 'sigmoid'}


Versiónn Final

In [ ]:
def walk_forward_fit_test(model_class, data, freq, search_space, model_params={}, n_trials=5):
    if freq == '1h':
        period = pd.Timedelta(days=7)
    elif freq == '4h':
        period = pd.Timedelta(days=15)
    else:
        period = pd.Timedelta(days=90)
    final_test_period = pd.Timedelta(days=365)

    def normalize_with_close(X, close_col):
        ratio_cols = [col for col in X.columns if any(x in col for x in ['sma', 'atr', 'min', 'max'])]
        for col in ratio_cols:
            X[col] = X[col] / close_col
        return X

    def prepare_features(df):
        df = df.copy()
        crypto_dummies = pd.get_dummies(df['crypto'], prefix='crypto')
        df = pd.concat([df.drop(columns=['crypto']), crypto_dummies], axis=1)
        return df, crypto_dummies.columns

    def objective(trial):
        # Construcción dinámica de parámetros desde el espacio definido
        trial_params = {}
        for param_name, param_info in search_space.items():
            if param_info['type'] == 'int':
                trial_params[param_name] = trial.suggest_int(param_name, *param_info['bounds'])
            elif param_info['type'] == 'float':
                trial_params[param_name] = trial.suggest_float(param_name, *param_info['bounds'], log=param_info.get('log', False))
            elif param_info['type'] == 'categorical':
                trial_params[param_name] = trial.suggest_categorical(param_name, param_info['choices'])
        # Añadir parámetros fijos (p.ej., random_state, class_weight, etc.)
        trial_params.update(model_params)

        # Preparación de datos
        df = data.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
        df.dropna(inplace=True)

        # Definición de periodos de validación
        max_time = df['timestamp'].max()
        cutoff = max_time - final_test_period
        df_trainval = df[df['timestamp'] < cutoff]

        # Generar fechas de split (últimos 5 períodos)
        min_time = df_trainval['timestamp'].min()
        split_dates = []
        current = min_time + period
        while current < cutoff:
            split_dates.append(current)
            current += period
        split_dates = split_dates[-5:]

        # Listas para métricas
        results_acc = []
        results_f1 = []

        for split_date in split_dates:
            # División train/test para este split
            train = df_trainval[df_trainval['timestamp'] < (split_date - pd.Timedelta(days=window_pred))]
            test  = df_trainval[(df_trainval['timestamp'] >= split_date) \
                                & (df_trainval['timestamp'] < split_date + period)]
            if len(test) == 0:
                continue

            # Columnas a eliminar (incluye timestamp, crypto y todas las _close)
            drop_cols = ['d', 'timestamp', 'crypto', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr'] \
                        + [c for c in train.columns if c.startswith('close')]
            X_train = train.drop(columns=drop_cols)
            y_train = train['d']
            X_test  = test.drop(columns=drop_cols)
            y_test  = test['d']

            # Normalización z-score
            mean, std = X_train.mean(), X_train.std()
            std.replace(0, 1, inplace=True)
            X_train = (X_train - mean) / std
            X_test  = (X_test  - mean) / std

            # Entrenamiento con sample_weight solo para GradientBoosting
            model = model_class(**trial_params)
            if model_class.__name__ == 'GradientBoostingClassifier':
                w = compute_sample_weight(class_weight='balanced', y=y_train)
                model.fit(X_train, y_train, sample_weight=w)
            else:
                model.fit(X_train, y_train)

            # Predicción y métricas
            preds = np.where(model.predict(X_test) > 0.5, 1, 0)
            results_acc.append(accuracy_score(y_test, preds))
            results_f1.append(f1_score(y_test, preds, zero_division=0))

        # Métricas promedio de validación
        avg_acc = np.mean(results_acc)
        avg_f1  = np.mean(results_f1)
        print(f'VALIDATION | acc={avg_acc:.4f} | f1={avg_f1:.4f}')
        return avg_f1


    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)

    best_params = study.best_params
    print("Mejores parámetros encontrados:", best_params)

    # Entrenamiento final con los mejores parámetros
    df = data.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['d'] = (df['close'].shift(-window_pred) > df['close']).astype(int)
    df.dropna(inplace=True)

    max_time = df['timestamp'].max()
    cutoff = max_time - final_test_period

    train = df[df['timestamp'] < (cutoff - pd.Timedelta(days=window_pred))]
    test = df[df['timestamp'] >= cutoff]

    if len(test) == 0:
        return best_params

    drop_cols = ['d', 'close', 'r', 'sma', 'min', 'max', 'mom', 'vol', 
                  'rsi', 'atr'] # close... se eliminan después de normalizar los datos con close_lag_1
    X_train_raw, y_train = train.drop(columns=drop_cols), train['d']
    X_test_raw, y_test = test.drop(columns=drop_cols), test['d']
    cryptos_test = test['crypto'].values

    # Normalizar
    close_train, close_test = train['close_lag_1'], test['close_lag_1']
    X_train_raw = normalize_with_close(X_train_raw.copy(), close_train)
    X_test_raw = normalize_with_close(X_test_raw.copy(), close_test)

    # One-hot encoding
    X_train, _ = prepare_features(X_train_raw)
    X_test, _ = prepare_features(X_test_raw)

    mean, std = X_train.mean(), X_train.std()
    std.replace(0, 1, inplace=True)
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    final_params = best_params.copy()
    if model_class.__name__ == "MLPClassifier" and "hidden_units" in final_params:
        hidden_units = final_params.pop("hidden_units")
        final_params["hidden_layer_sizes"] = (hidden_units,)
    model = model_class(**final_params)
    
    if model_class.__name__ == "GradientBoostingClassifier":
        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
        model.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train, y_train)

    pred = np.where(model.predict(X_test) > 0.5, 1, 0)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    print(f'FINAL TEST | acc={acc:.4f} | f1={f1:.4f}')
    print(f'desbalanceo de los datos reales {y_test.value_counts(normalize=True)}')
    print(f'desbalanceo de las predicciones {pd.Series(pred).value_counts(normalize=True)}')
    save_results_global(model_class.__name__, "global", acc, "FINAL TEST", frequency=freq)
    save_results_global(model_class.__name__, "global", f1, "F1 FINAL TEST", frequency=freq)

    # Preparar DataFrame de resultados por criptomoneda
    df_results_test = X_test.copy()
    df_results_test['true'] = y_test.values
    df_results_test['pred'] = pred
    df_results_test['crypto'] = cryptos_test

    # Métricas por criptomoneda
    print("\nFINAL TEST - Accuracy y F1 por criptomoneda:")
    grouped = df_results_test.groupby('crypto')
    for crypto, group in grouped:
        acc_c = accuracy_score(group['true'], group['pred'])
        f1_c = f1_score(group['true'], group['pred'], zero_division=0)  # evita error si solo hay una clase
        print(f"{crypto:<15} | acc = {acc_c:.4f} | f1 = {f1_c:.4f}")
        print(f"    Desbalanceo reales      : {group['true'].value_counts(normalize=True).to_dict()}")
        print(f"    Desbalanceo predicciones: {group['pred'].value_counts(normalize=True).to_dict()}")
        save_results_global(model_class.__name__, crypto, acc_c, "CRYPTO TEST", frequency=freq)
        save_results_global(model_class.__name__, crypto, f1_c, "F1 CRYPTO TEST", frequency=freq)

    return final_params


In [ ]:
# === Definición del espacio de búsqueda para cada modelo ===

search_spaces = {
    "MLPClassifier": {
        "hidden_layer_sizes": {"type": "int",   "bounds": (32, 1024), "step": 32},
        "alpha":              {"type": "float", "bounds": (1e-6, 1e-1), "log": True},
        "learning_rate_init": {"type": "float", "bounds": (1e-5, 1e-1), "log": True},
    },
    "RandomForestClassifier": {
        "n_estimators":      {"type": "int",         "bounds": (100, 1000), "step": 100},
        "max_depth":         {"type": "int",         "bounds": (3,   30)},
        "min_samples_split": {"type": "int",         "bounds": (2,   10)},
        "min_samples_leaf":  {"type": "int",         "bounds": (1,   10)},
        "max_features":      {"type": "categorical", "choices": ["sqrt", "log2", None]},
        "bootstrap":         {"type": "categorical", "choices": [True, False]},
    },
    "SVC": {
        "C":     {"type": "float", "bounds": (1e-2, 1e3), "log": True},
        "gamma": {"type": "float", "bounds": (1e-5, 1e-1), "log": True},
        "kernel":{"type": "categorical", "choices": ["rbf", "poly", "sigmoid"]},
    },
    "GradientBoostingClassifier": {
        "n_estimators":      {"type": "int",   "bounds": (50, 500),  "step": 50},
        "learning_rate":     {"type": "float", "bounds": (1e-3, 0.3), "log": True},
        "max_depth":         {"type": "int",   "bounds": (3,   15)},
        "min_samples_split": {"type": "int",   "bounds": (2,   20)},
        "min_samples_leaf":  {"type": "int",   "bounds": (1,   20)},
    },
}

# === Parámetros fijos para cada modelo ===

model_fixed_params = {
    "MLPClassifier": {
        "max_iter": 1000,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "shuffle": False,
        "random_state": 100,
    },
    "RandomForestClassifier": {
        "class_weight": "balanced",
        "random_state": 100,
        "n_jobs": -1
    },
    "SVC": {
        "probability": True,
        "class_weight": "balanced",
        "random_state": 100
    },
    "GradientBoostingClassifier": {
        "random_state": 100
    }
}

# === Diccionario de clases de modelos ===

model_classes = {
    "MLPClassifier": MLPClassifier,
    "RandomForestClassifier": RandomForestClassifier,
    "SVC": SVC,
    "GradientBoostingClassifier": GradientBoostingClassifier
}

# === Entrenamiento en bucle ===

best_params_dict = {}

for model_name, model_cls in model_classes.items():
    print(f"\n\n=== Entrenando modelo: {model_name} ===\n")
    
    best_params = walk_forward_fit_test(
        model_class=model_cls,
        data=df_global,
        freq=frequency,
        search_space=search_spaces[model_name],
        model_params=model_fixed_params.get(model_name, {}),
        n_trials=10  
    )

    best_params_dict[model_name] = best_params

print("\n\n=== Mejores hiperparámetros por modelo ===")
for model_name, params in best_params_dict.items():
    print(f"{model_name}: {params}")


[I 2025-06-01 11:39:23,754] A new study created in memory with name: no-name-3d6358b5-d3ec-46aa-8146-c290ddcf38ce




=== Entrenando modelo: MLPClassifier ===



[I 2025-06-01 11:39:56,681] Trial 4 finished with value: 0.749046904633939 and parameters: {'hidden_layer_sizes': 53, 'alpha': 3.137037565142582e-05, 'learning_rate_init': 0.003644127702266552}. Best is trial 4 with value: 0.749046904633939.


VALIDATION | acc=0.7127 | f1=0.7490


[I 2025-06-01 11:43:31,181] Trial 9 finished with value: 0.748159898050669 and parameters: {'hidden_layer_sizes': 497, 'alpha': 0.0817320694253442, 'learning_rate_init': 0.0020597944355763165}. Best is trial 4 with value: 0.749046904633939.


...

VALIDATION | acc=0.6586 | f1=0.7327


[I 2025-06-01 12:04:39,509] Trial 3 finished with value: 0.7536064655240322 and parameters: {'C': 0.2656849885482458, 'gamma': 0.09324105193045461, 'kernel': 'poly'}. Best is trial 1 with value: 0.7555531193680401.


VALIDATION | acc=0.7112 | f1=0.7536


[I 2025-06-01 12:19:25,343] Trial 0 finished with value: 0.743453714410769 and parameters: {'C': 74.18483566518019, 'gamma': 0.041565311160406695, 'kernel': 'poly'}. Best is trial 1 with value: 0.7555531193680401.


VALIDATION | acc=0.7047 | f1=0.7435


In [ ]:
model_names = []
real_0, real_1, pred_0, pred_1 = [], [], [], []

for model, res in data_graf_dict.items():
    y_test = pd.Series(res["true"])
    y_pred = pd.Series(res["pred"])
    
    real_dist = y_test.value_counts(normalize=True).to_dict()
    pred_dist = y_pred.value_counts(normalize=True).to_dict()

    model_names.append(model)
    real_0.append(real_dist.get(0, 0))
    real_1.append(real_dist.get(1, 0))
    pred_0.append(pred_dist.get(0, 0))
    pred_1.append(pred_dist.get(1, 0))

# Posiciones para las barras
x = np.arange(len(model_names))
bar_width = 0.2

# Crear gráfico
fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(x - 1.5*bar_width, real_0, width=bar_width, label='Clase 0 - Real', color='tab:blue', alpha=0.7)
ax.bar(x - 0.5*bar_width, pred_0, width=bar_width, label='Clase 0 - Predicho', color='tab:blue', alpha=0.3)

ax.bar(x + 0.5*bar_width, real_1, width=bar_width, label='Clase 1 - Real', color='tab:orange', alpha=0.7)
ax.bar(x + 1.5*bar_width, pred_1, width=bar_width, label='Clase 1 - Predicho', color='tab:orange', alpha=0.3)

# Etiquetas y estilo
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=45, ha='right')
ax.set_ylabel('Proporción')
ax.set_ylim(0, 1)
ax.set_title('Comparación de Proporciones Reales vs Predichas por Clase y Modelo')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.cm as cm
df_all = []
for model_name, results in desb_graf_dict.items():
    df_model = pd.DataFrame({
        "crypto": results["crypto"],
        "acc": results["acc"],
        "f1": results["f1"],
        "model": model_name
    })
    df_all.append(df_model)

df = pd.concat(df_all, ignore_index=True)

# Preparar gráfico
cryptos = df["crypto"].unique()
models = df["model"].unique()
metrics = ["acc", "f1"]

fig, ax = plt.subplots(figsize=(16, 6))
bar_width = 0.05
x = np.arange(len(cryptos))

# Paleta de colores base por modelo
colors = cm.get_cmap('tab10', len(models))  # tab10 da 10 colores distintos

# Calcular desplazamientos
total_bars = len(models) * len(metrics)
offsets = np.linspace(-bar_width * total_bars / 2, bar_width * total_bars / 2, total_bars)

for i, model in enumerate(models):
    color_base = colors(i)
    for j, metric in enumerate(metrics):
        values = df[df["model"] == model].set_index("crypto").loc[cryptos][metric]
        alpha = 1.0 if metric == "acc" else 0.4  # Misma base de color, más claro para F1
        label = f"{model} - {'Accuracy' if metric == 'acc' else 'F1 Score'}"
        position = x + offsets[i * len(metrics) + j]
        ax.bar(position, values, width=bar_width, label=label, color=color_base, alpha=alpha)

# Ejes
ax.set_xticks(x)
ax.set_xticklabels(cryptos, rotation=45)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.set_title("Accuracy y F1 Score por Criptomoneda y Modelo (Mismo color por modelo)")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='Límite 0.5')
plt.tight_layout()
plt.show()
